# Task 02: RAG Pipeline

Build a Retrieval-Augmented Generation pipeline over the company knowledge base. The pipeline should answer customer questions grounded in the knowledge base articles.

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Build the Vector Store

Create a FAISS vector store from the knowledge base articles using OpenAI embeddings. Create a retriever that returns the top 3 most relevant documents.

In [ ]:
# YOUR CODE HERE
# Define: embeddings_model, vectorstore, retriever



In [ ]:
# TEST — Structural check (no API call)
assert 'retriever' in dir() or 'retriever' in locals(), "Define a variable named 'retriever'"
assert hasattr(retriever, 'invoke'), "retriever must have .invoke() method"
print("✓ retriever defined and callable")


In [ ]:
# TEST — Retriever returns relevant results
refund_docs = retriever.invoke("How long does a refund take?")

assert len(refund_docs) > 0, "Retriever returned no documents"
assert len(refund_docs) <= 5, "Retrieved too many documents (expected k=3)"

titles = [doc.metadata.get("title", "") for doc in refund_docs]
print(f"Retrieved {len(refund_docs)} docs: {titles}")
assert any("refund" in t.lower() or "policy" in t.lower() for t in titles),     "Refund article not retrieved for refund query — check your index"
print("✓ Retriever returns relevant documents")


## Part 2: Build the RAG Chain

Create a `rag_chain` using LCEL that:
- Takes a question string as input
- Retrieves relevant articles
- Generates a grounded answer

Format retrieved documents before inserting them into the prompt.

In [ ]:
# YOUR CODE HERE
# Define: format_docs(docs) -> str, rag_prompt, rag_chain



In [ ]:
# TEST — RAG chain produces answers
answer = rag_chain.invoke("How long does a refund take to process?")

assert isinstance(answer, str), "rag_chain must return a string"
assert len(answer) > 20, "Answer too short — the chain may not be working"

answer_lower = answer.lower()
found = any(kw in answer_lower for kw in ["5-7", "business days", "refund"])
print(f"Answer: {answer[:200]}")
assert found, "Answer doesn't mention refund timeline — check your retriever and prompt"
print("✓ RAG chain produces relevant answers")


## Part 3: Evaluate Against All Test Questions

Run the RAG chain over all 5 test questions and check keyword coverage.

In [ ]:
# YOUR CODE HERE
# Implement evaluate_rag(rag_chain, test_questions) -> float
# Returns average keyword coverage (0.0 to 1.0)



In [ ]:
# TEST — Evaluation over all test questions
coverage = evaluate_rag(rag_chain, test_questions)

print(f"\nOverall keyword coverage: {coverage:.0%}")

for q in test_questions:
    answer = rag_chain.invoke(q["question"]).lower()
    hits = [kw for kw in q["expected_keywords"] if kw.lower() in answer]
    score = len(hits) / len(q["expected_keywords"])
    status = "✓" if score >= 0.5 else "✗"
    print(f"  {status} Q{q['id']}: {q['question'][:50]!r} — {score:.0%} ({hits})")

assert coverage >= 0.60, f"RAG quality {coverage:.0%} < 60% — improve retrieval or prompt"
print(f"\n✓ RAG evaluation passed ({coverage:.0%} coverage)")
